![Cloud-First](../image/CloudFirst.png)

# SIT742: Modern Data Science
**(Module 02: Python Foundations for Big Data)**

**Session 2F: File Usage**

---

- Materials in this module have been developed to support practical learning in modern data science, big data processing, and applied analytics.
- Materials may include adapted or referenced open-source resources. Keep attribution and licence notes where applicable.
- The public notebook collection is available in [SIT742](https://github.com/tulip-lab/sit742).
- If you find any issue or bug in this document, please submit an issue at [SIT742](https://github.com/tulip-lab/sit742/issues).
- Audience: Honours and Master's students using the Deakin SIT742 practical and self-learning materials.

Prepared by the SIT742 Teaching Team.

Maintained through the [TULIP Lab](https://www.tulip.academy) FLIP workflow.

---

## Session 2F: File Usage

<div align="center">

<table>
<thead>
<tr>
<th><strong>Item</strong></th>
<th><strong>Description</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Module context</td>
<td>This notebook is one component of the practical and self-learning materials for this module. The full module is normally completed across two two-hour practical sessions, together with the other notebooks listed in the SIT742 repository.</td>
</tr>
<tr>
<td align="left">Environment</td>
<td>Google Colab or local Jupyter with standard Python 3</td>
</tr>
<tr>
<td align="left">Main output</td>
<td>Small text and CSV files read from public teaching data and written to a temporary working folder.</td>
</tr>
<tr>
<td align="left">Related assessment</td>
<td>Not directly assessed</td>
</tr>
</tbody>
</table>

</div>

---

**Table of Contents**

- [1. Overview and Learning Goals](#1-overview-and-learning-goals)
- [2. Setup and Background](#2-setup-and-background)
- [3. Core Concepts](#3-core-concepts)
- [4. Guided Examples](#4-guided-examples)
- [5. Testing and Analysis](#5-testing-and-analysis)
- [6. Student Tasks](#6-student-tasks)
- [7. Reflection and References](#7-reflection-and-references)


<a id="1-overview-and-learning-goals"></a>

### 1. Overview and Learning Goals

Many data science workflows begin by reading data from files and end by writing cleaned or transformed data back to disk. This notebook introduces text files, CSV files, file paths, context managers, and small reproducible examples.

The original notebook downloaded `score.txt` and `score.csv` with the third-party `wget` package, then wrote output files in the notebook directory. This version uses the public data files already included in the SIT742 repository when available, falls back to an in-memory teaching copy when needed, and writes generated files to a temporary working folder. This keeps the examples runnable without package installation and avoids leaving generated files in the repository.

By the end of this notebook, you should be able to:

1. locate a data file with `pathlib.Path` and a relative path;
2. read a text file with `with open(...)` and split fields safely;
3. write a small text output file with explicit newline handling;
4. read and write CSV files with Python's `csv` module;
5. explain why encoding, newline handling, and working directories matter.


<a id="2-setup-and-background"></a>

### 2. Setup and Background

This notebook uses only standard Python. It looks for `score.txt` and `score.csv` in the public `Jupyter/data/` folder. If the data folder is not available in your environment, the setup cell creates a small teaching copy inside a temporary working directory.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import csv
import sys

print('Python version:', sys.version.split()[0])
print('Current working directory:', Path.cwd())

_WORK_DIR_MANAGER = TemporaryDirectory()
WORK_DIR = Path(_WORK_DIR_MANAGER.name)

SAMPLE_TEXT = '''David 3402 80
Jane 3403 76
Sophia 3405 65
Jane 3447 92
William 3456 75
'''
SAMPLE_CSV = '''Name,ID,Score
David,3402,80
Jane,3403,76
Sophia,3405,65
Jane,3447,92
William,3456,75
'''

def find_data_dir():
    candidates = [
        (Path.cwd() / '../data').resolve(),
        (Path.cwd() / 'Jupyter/data').resolve(),
        (Path.cwd().parent / 'data').resolve(),
        (Path.cwd() / 'data').resolve(),
    ]
    for candidate in candidates:
        if (candidate / 'score.txt').exists() and (candidate / 'score.csv').exists():
            return candidate, 'public SIT742 repository data'

    fallback = WORK_DIR / 'sample-data'
    fallback.mkdir(parents=True, exist_ok=True)
    (fallback / 'score.txt').write_text(SAMPLE_TEXT, encoding='utf-8')
    (fallback / 'score.csv').write_text(SAMPLE_CSV, encoding='utf-8')
    return fallback, 'temporary teaching copy'

DATA_DIR, DATA_SOURCE = find_data_dir()
print('Data source:', DATA_SOURCE)
print('Data directory:', DATA_DIR)
print('Temporary output directory:', WORK_DIR)


<a id="3-core-concepts"></a>

### 3. Core Concepts

<div align="center">

<table>
<thead>
<tr>
<th><strong>Concept</strong></th>
<th><strong>Meaning</strong></th>
<th><strong>Why it matters</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Path</td>
<td align="left">A location for a file or folder.</td>
<td align="left">Notebooks can run from different working directories, so paths should be explicit and relative where possible.</td>
</tr>
<tr>
<td align="left">Mode</td>
<td align="left"><code>'r'</code>, <code>'w'</code>, or <code>'a'</code> controls reading, writing, or appending.</td>
<td align="left"><code>'w'</code> replaces existing file contents.</td>
</tr>
<tr>
<td align="left">Context manager</td>
<td align="left"><code>with open(...) as f:</code> opens a file and closes it automatically.</td>
<td align="left">Avoids forgotten <code>close()</code> calls.</td>
</tr>
<tr>
<td align="left">Encoding</td>
<td align="left">The rule used to turn bytes into text.</td>
<td align="left">Using <code>encoding='utf-8'</code> makes text reading more reproducible.</td>
</tr>
<tr>
<td align="left">CSV</td>
<td align="left">Comma-separated values, a common table format.</td>
<td align="left">The <code>csv</code> module handles rows and delimiters more safely than manual splitting.</td>
</tr>
</tbody>
</table>

</div>


<a id="4-guided-examples"></a>

### 4. Guided Examples

#### 4.1 Reading Text Files

The teaching data file `score.txt` contains one record per line:

```text
Name StudentID Score
David 3402 80
Jane 3403 76
```

Use `Path` objects to build file paths. Use a `with` block so the file is closed automatically.


In [ ]:
score_txt_path = DATA_DIR / 'score.txt'
print(score_txt_path)

with score_txt_path.open('r', encoding='utf-8') as scorefile:
    for line in scorefile:
        value = line.split()
        name = value[0]
        student_id = value[1]
        score = value[2]
        print('%s with ID %s has a score of %s' % (name, student_id, score))


The original example used manual `open()` and `close()` calls. The `with` pattern is preferred because Python closes the file even if an error occurs inside the block.

#### 4.2 Writing Text Files

The next example writes a derived file containing only student ID and score. The output goes into the temporary working folder printed in the setup cell.


In [ ]:
id_txt_path = WORK_DIR / 'id.txt'

with score_txt_path.open('r', encoding='utf-8') as infile, id_txt_path.open('w', encoding='utf-8') as outfile:
    for line in infile:
        values = line.split()
        student_id = values[1]
        score = values[2]
        dataline = student_id + ',' + score
        outfile.write(dataline + '\n')

print('Wrote:', id_txt_path)


You can read the generated text file to inspect the result.


In [ ]:
message = id_txt_path.read_text(encoding='utf-8')
print(message)


#### 4.3 Reading CSV Files

CSV files are text files with a tabular structure. The `csv.reader()` function turns each row into a list of strings.


In [ ]:
score_csv_path = DATA_DIR / 'score.csv'

with score_csv_path.open('r', encoding='utf-8', newline='') as infile:
    incsv = csv.reader(infile, delimiter=',')
    header = next(incsv)
    print('Header:', header)
    for row in incsv:
        name, student_id, score = row
        print('%s %s %s' % (student_id, name, score))


#### 4.4 Writing CSV Files

Use `csv.writer()` to write rows. Passing `newline=''` avoids extra blank lines on some platforms.


In [ ]:
id_csv_path = WORK_DIR / 'id.csv'

with score_csv_path.open('r', encoding='utf-8', newline='') as infile, id_csv_path.open('w', encoding='utf-8', newline='') as outfile:
    incsv = csv.reader(infile, delimiter=',')
    outcsv = csv.writer(outfile, delimiter=',')
    header = next(incsv)
    outcsv.writerow(['ID', 'Score'])
    for row in incsv:
        name, student_id, score = row
        row_output = [student_id, score]
        outcsv.writerow(row_output)

print('Wrote:', id_csv_path)
print(id_csv_path.read_text(encoding='utf-8'))


<a id="5-testing-and-analysis"></a>

### 5. Testing and Analysis

Use visible checks to confirm that the text and CSV workflows produced the expected shape.


In [ ]:
assert score_txt_path.exists()
assert score_csv_path.exists()
assert id_txt_path.exists()
assert id_csv_path.exists()

text_lines = id_txt_path.read_text(encoding='utf-8').strip().splitlines()
assert len(text_lines) == 5
assert all(',' in line for line in text_lines)

with id_csv_path.open('r', encoding='utf-8', newline='') as infile:
    rows = list(csv.reader(infile))

assert rows[0] == ['ID', 'Score']
assert len(rows) == 6
assert rows[1] == ['3402', '80']

print('File and CSV checks passed.')


Interpretation prompts:

1. Which files are inputs and which files are generated outputs?
2. Why is the generated output written to a temporary folder?
3. What would change if the input file contained a name with a space in it?


<a id="6-student-tasks"></a>

### 6. Student Tasks

Complete these tasks in new cells using the same `DATA_DIR` and `WORK_DIR` variables.

<div align="center">

<table>
<thead>
<tr>
<th><strong>Task</strong></th>
<th><strong>What you need to do</strong></th>
<th><strong>Why it matters</strong></th>
<th><strong>Expected evidence</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Task 1</td>
<td align="left">Read `score.txt` and count how many records it contains.</td>
<td align="left">Practise line iteration and simple counters.</td>
<td align="left">Printed record count.</td>
</tr>
<tr>
<td align="left">Task 2</td>
<td align="left">Write a text file containing only records with scores of at least 75.</td>
<td align="left">Combine file reading, conditions, and file writing.</td>
<td align="left">Generated file path and printed file contents.</td>
</tr>
<tr>
<td align="left">Task 3</td>
<td align="left">Read `score.csv` with `csv.DictReader()` and print each row as a dictionary.</td>
<td align="left">Understand a more readable CSV access pattern.</td>
<td align="left">Dictionary output for each row.</td>
</tr>
<tr>
<td align="left">Task 4</td>
<td align="left">Explain why generated files should not overwrite the original data files.</td>
<td align="left">Develop reproducible workflow habits.</td>
<td align="left">Short written explanation.</td>
</tr>
</tbody>
</table>

</div>


In [ ]:
# Student workspace for M02F tasks.
# Use DATA_DIR for input files and WORK_DIR for generated output files.
print('Create your M02F task solutions in new cells below this prompt.')


<a id="7-reflection-and-references"></a>

### 7. Reflection and References

Reflection questions:

1. How does using `with open(...)` reduce file-handling mistakes?
2. What information would you need before reusing a dataset outside this unit?
3. Why is `csv.reader()` safer than splitting CSV rows manually with `str.split(',')`?

#### Further Readings

- [Python documentation: pathlib](https://docs.python.org/3/library/pathlib.html)
- [Python documentation: open](https://docs.python.org/3/library/functions.html#open)
- [Python documentation: csv](https://docs.python.org/3/library/csv.html)
- [Public SIT742 repository](https://github.com/tulip-lab/sit742)

Dataset note: `score.txt` and `score.csv` are small teaching data files distributed with the public SIT742 repository for notebook practice. Check repository and dataset-specific licence information before reusing data outside this unit context.
